<a href="https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb pandas pyarrow huggingface_hub fsspec

In [2]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")

login(token=HF_TOKEN)

print("Hugging Face login successful")

Hugging Face login successful


In [3]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

print("DuckDB authenticated")


DuckDB authenticated


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline rule

This baseline identifies content pages that may benefit from manual review using only information that is available at prediction time.

The rule assigns higher scores to pages that have:

- low Google Search Console CTR,
- high search impressions,
- an average search position between 5 and 20.

Pages with higher scores are considered stronger candidates for review because they may represent important pages with unrealized search potential.

This rule does not use future performance, prediction labels, or any information unavailable at scoring time.

## Reason codes

- **CTR_FIX** – Low CTR on pages with average search positions between 5 and 20, indicating an opportunity to improve click-through rates.
- **CTR_FIX_REFRESH** – Low CTR on high-impression pages that should be prioritized for review.
- **REVIEW** – General review recommendation when no specific rule is triggered.

In [4]:
query = """
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
LIMIT 1
"""

sample = con.execute(query).df()

print(sample.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Baseline action score

This baseline ranks pages using a simple rule-based scoring system based on three signals available at prediction time:

- High search impressions (greater opportunity for impact)
- Low Google Search Console CTR (potential CTR improvement)
- Average search position between 5 and 20 (good ranking with room for improvement)

The score is the sum of these conditions. Each page is assigned a reason code and an action label based on the signals that triggered the score. The ranked queue is written to `work/outputs/baseline_action_score.csv`.

Reason codes:
- `CTR_FIX` – Low CTR with a ranking position that suggests CTR improvement opportunities.
- `CTR_FIX_REFRESH` – Low CTR on pages with high impressions, making them high-priority review candidates.
- `REVIEW` – General review recommendation when no specific rule is triggered.

In [5]:
import pandas as pd
import os

query = """
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'
)
WHERE gsc_data_available = TRUE
"""

df = con.execute(query).df()

df["gsc_ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].replace(0, pd.NA)
) * 100

df["gsc_ctr"] = df["gsc_ctr"].fillna(0)

df["score"] = 0

df.loc[df["gsc_impressions"] >= 100, "score"] += 30
df.loc[df["gsc_ctr"] < 2, "score"] += 40
df.loc[df["gsc_avg_position"].between(5, 20), "score"] += 30

df["reason_code"] = "REVIEW"

df.loc[
    (df["gsc_ctr"] < 2) &
    (df["gsc_avg_position"].between(5, 20)),
    "reason_code"
] = "CTR_FIX"

df.loc[
    (df["gsc_ctr"] < 2) &
    (df["gsc_impressions"] >= 100),
    "reason_code"
] = "CTR_FIX_REFRESH"

df["action_label"] = "Review Page"

df.loc[df["reason_code"] == "CTR_FIX", "action_label"] = "Improve CTR"
df.loc[df["reason_code"] == "CTR_FIX_REFRESH", "action_label"] = "Improve CTR & Review"

df = df.sort_values(
    by=["score", "gsc_impressions"],
    ascending=[False, False]
)

os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print(f"Rows scored: {len(df):,}")

df.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows scored: 3,878,937


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_ctr,score,reason_code,action_label
962777,2026-06-11,client_e547b89c05043229,content_963de14b1f58978f,245826,890,6.326149,0.362045,100,CTR_FIX_REFRESH,Improve CTR & Review
1239844,2026-06-12,client_e547b89c05043229,content_963de14b1f58978f,46220,81,6.856274,0.175249,100,CTR_FIX_REFRESH,Improve CTR & Review
3877299,2026-06-30,client_06d356715a8ff3b6,content_f88878f155e4838d,45890,329,5.828089,0.716932,100,CTR_FIX_REFRESH,Improve CTR & Review
3785689,2026-06-29,client_06d356715a8ff3b6,content_f88878f155e4838d,35560,313,5.676069,0.880202,100,CTR_FIX_REFRESH,Improve CTR & Review
3017750,2026-06-24,client_e547b89c05043229,content_0ec99ef7d7e11565,30645,50,5.019710,0.163159,100,CTR_FIX_REFRESH,Improve CTR & Review
3154685,2026-06-28,client_06d356715a8ff3b6,content_f88878f155e4838d,28945,182,5.951736,0.628779,100,CTR_FIX_REFRESH,Improve CTR & Review
2785755,2026-06-22,client_a80fca3f171ed1de,content_012de75c008aa653,25191,0,7.027033,0.000000,100,CTR_FIX_REFRESH,Improve CTR & Review
1274146,2026-06-08,client_fef1a8f436438636,content_ba462518dad435fc,25111,3,13.825375,0.011947,100,CTR_FIX_REFRESH,Improve CTR & Review
2147561,2026-06-18,client_e547b89c05043229,content_963de14b1f58978f,24435,49,6.485697,0.200532,100,CTR_FIX_REFRESH,Improve CTR & Review
2020062,2026-06-14,client_8ddc46da5414ffd8,content_b902320872acab45,23358,6,5.213503,0.025687,100,CTR_FIX_REFRESH,Improve CTR & Review


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 review

The top 20 ranked pages were selected because they have high search visibility but relatively low click-through rates, making them strong candidates for manual review. Most of these pages received the `CTR_FIX_REFRESH` reason code, indicating that they combine high impressions with low CTR and may have opportunities to improve search performance.

These recommendations are intended for decision support rather than automatic action. Each page should be reviewed manually to confirm that the recommendation is appropriate, since factors such as search intent, seasonality, branded searches, or strong competition may explain the observed performance.

In [6]:
top20 = df.head(20)

for i, row in enumerate(top20.itertuples(index=False), start=1):
    print(f"{i}.")
    print(f"Content: {row.content_hash_id}")
    print(f"Action: {row.action_label}")
    print(f"Reason Code: {row.reason_code}")
    print("Confidence: High - High impressions with low CTR.")
    print("What would make it wrong: Low CTR is caused by search intent, seasonality, branded searches, or strong competitors rather than page quality.")
    print("-" * 80)

1.
Content: content_963de14b1f58978f
Action: Improve CTR & Review
Reason Code: CTR_FIX_REFRESH
Confidence: High - High impressions with low CTR.
What would make it wrong: Low CTR is caused by search intent, seasonality, branded searches, or strong competitors rather than page quality.
--------------------------------------------------------------------------------
2.
Content: content_963de14b1f58978f
Action: Improve CTR & Review
Reason Code: CTR_FIX_REFRESH
Confidence: High - High impressions with low CTR.
What would make it wrong: Low CTR is caused by search intent, seasonality, branded searches, or strong competitors rather than page quality.
--------------------------------------------------------------------------------
3.
Content: content_f88878f155e4838d
Action: Improve CTR & Review
Reason Code: CTR_FIX_REFRESH
Confidence: High - High impressions with low CTR.
What would make it wrong: Low CTR is caused by search intent, seasonality, branded searches, or strong competitors rather

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks + leakage check

Some recommendations may be weak because a low CTR does not always indicate a page problem. Search intent, seasonal trends, branded searches, or strong competitors can reduce CTR even when the content is performing as expected. These pages should therefore be reviewed manually before any action is taken.

This baseline uses only signals that are available at scoring time, including search impressions, clicks, CTR, and average position. It does not use future performance, prediction labels, or product-generated flags, so no data leakage is introduced into the scoring rule.

In [7]:
print("Highest score:", df["score"].max())
print("Lowest score:", df["score"].min())

print("\nReason code distribution:")
print(df["reason_code"].value_counts())

print("\nAction label distribution:")
print(df["action_label"].value_counts())

Highest score: 100
Lowest score: 0

Reason code distribution:
reason_code
REVIEW             2038754
CTR_FIX            1416916
CTR_FIX_REFRESH     423267
Name: count, dtype: int64

Action label distribution:
action_label
Review Page             2038754
Improve CTR             1416916
Improve CTR & Review     423267
Name: count, dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.